# 04 - Feature Engineering

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_config
from src.utils import setup_logging, ensure_dir
from src.feature_engineering import (
    create_temporal_derivatives, create_band_ratios,
    create_vi_interactions, create_agronomic_indices,
    create_vi_soil_interactions, create_phenological_features,
    create_seasonal_aggregates, engineer_all_features,
)
from src.saxton_rawls import add_saxton_features
from src.temporal_alignment import generate_period_windows

logger = setup_logging()
config = load_config()
output_dir = config['data']['output_dir']

AGGREGATION_MODE = config['aggregation']['mode']
print(f"Aggregation mode: {AGGREGATION_MODE}")

Aggregation mode: peak_stages


## 1. Load Aggregated Features

In [2]:
features_df = pd.read_parquet(f'{output_dir}/features_{AGGREGATION_MODE}.parquet')
print(f"Input features: {features_df.shape}")

# Load double logistic params for phenological features
try:
    params_df = pd.read_csv(f'{output_dir}/double_logistic_params.csv')
    print(f"Double logistic params: {len(params_df)} fields")
except FileNotFoundError:
    params_df = pd.DataFrame()
    print("No double logistic params found (phenological features will be skipped)")

# Get period names
config['aggregation']['mode'] = AGGREGATION_MODE
windows = generate_period_windows(config)
period_names = [w[0] for w in windows]
print(f"Periods: {period_names}")

Input features: (228, 806)
Double logistic params: 240 fields
Periods: ['cp1', 'cp2', 'cp3', 'cp4', 'cp5']


## 2. Saxton-Rawls Soil Features

In [3]:
# Check if Saxton-Rawls features already exist
sr_cols = [c for c in features_df.columns if c.startswith('saxton_')]
if sr_cols:
    print(f"Saxton-Rawls features already present: {len(sr_cols)}")
else:
    # Check for soil texture columns
    has_sand = 'soil_top_sand' in features_df.columns
    has_clay = 'soil_top_clay' in features_df.columns
    has_om = 'soil_top_om' in features_df.columns
    
    if has_sand and has_clay and has_om:
        features_df = add_saxton_features(features_df)
        sr_cols = [c for c in features_df.columns if c.startswith('saxton_')]
        print(f"Added {len(sr_cols)} Saxton-Rawls features")
    else:
        print(f"Missing soil texture columns (sand={has_sand}, clay={has_clay}, om={has_om})")

13:53:01 | INFO    | Saxton-Rawls: 228/228 fields with valid soil data


Added 11 Saxton-Rawls features


## 3. Run Full Feature Engineering Pipeline

In [4]:
engineered_df = engineer_all_features(
    features_df, period_names, config, params_df=params_df,
)

print(f"\nFeature engineering summary:")
print(f"  Input columns: {features_df.shape[1]}")
print(f"  Output columns: {engineered_df.shape[1]}")
print(f"  New features: {engineered_df.shape[1] - features_df.shape[1]}")
print(f"  Fields: {len(engineered_df)}")

c:\Users\Vlasis\Repos\GPC_Pipeline\notebooks\..\src\feature_engineering.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{feat}_ratio_{p1}_{p2}"] = ratio.clip(-10, 10)
c:\Users\Vlasis\Repos\GPC_Pipeline\notebooks\..\src\feature_engineering.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{feat}_roc_{p1}_{p2}"] = roc.clip(-10, 10)
c:\Users\Vlasis\Repos\GPC_Pipeline\notebooks\..\src\feature_engineering.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inser


Feature engineering summary:
  Input columns: 817
  Output columns: 1491
  New features: 674
  Fields: 228


## 4. Feature Summary

In [5]:
# Categorize features
cols = [c for c in engineered_df.columns if c != 'field_key']
categories = {
    'Temporal (raw)': [c for c in cols if any(c.startswith(p + '_') for p in period_names) and '_change_' not in c and '_ratio_' not in c and '_roc_' not in c and '_div_' not in c and '_x_' not in c],
    'Change features': [c for c in cols if '_change_' in c],
    'Ratio features': [c for c in cols if '_ratio_' in c],
    'Rate of change': [c for c in cols if '_roc_' in c],
    'Band ratios': [c for c in cols if '_div_' in c],
    'VI interactions': [c for c in cols if '_x_' in c and 'soil' not in c and 'pheno' not in c],
    'VI x Soil': [c for c in cols if '_x_soil' in c or '_x_soil_' in c],
    'Agronomic': [c for c in cols if 'aridity' in c or 'water_deficit' in c or 'heat_stress' in c],
    'Phenological': [c for c in cols if c.startswith('pheno_')],
    'Seasonal': [c for c in cols if 'seasonal_' in c],
    'Soil (static)': [c for c in cols if 'soil' in c.lower() and '_x_' not in c],
    'Saxton-Rawls': [c for c in cols if c.startswith('saxton_')],
}

print(f"{'Category':<25} {'Count':>8}")
print("-" * 35)
accounted = set()
for cat, feats in categories.items():
    print(f"{cat:<25} {len(feats):>8}")
    accounted.update(feats)
other = set(cols) - accounted
print(f"{'Other':<25} {len(other):>8}")
print(f"{'TOTAL':<25} {len(cols):>8}")

Category                     Count
-----------------------------------
Temporal (raw)                 735
Change features                168
Ratio features                 196
Rate of change                 168
Band ratios                     25
VI interactions                 67
VI x Soil                       24
Agronomic                       20
Phenological                     9
Seasonal                        30
Soil (static)                   43
Saxton-Rawls                    11
Other                           22
TOTAL                         1490


## 5. Save Engineered Features

In [6]:
output_path = f'{output_dir}/engineered_features_{AGGREGATION_MODE}.parquet'
engineered_df.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {engineered_df.shape}")
print(f"\nNotebook 04 complete!")

Saved: data/processed/engineered_features_peak_stages.parquet
Shape: (228, 1491)

Notebook 04 complete!
